# 请求后端 LangGraph Platform 流式接口

本 Notebook 通过 `langgraph-sdk` 直接请求本地 LangGraph Platform 服务（`http://localhost:8123`），与前端 `useAgentChat` 调用的是同一套后端。

- Assistant ID：`paas-agent`
- 流式端点：`/runs/stream`（SDK 自动拼接）
- StreamMode：`values` + `events`

In [ ]:
from langgraph_sdk import get_client

API_URL = "http://localhost:8123"
ASSISTANT_ID = "paas-agent"

client = get_client(url=API_URL)
client

## 1. 检查可用 assistants

In [ ]:
assistants = await client.assistants.search()
for a in assistants:
    print(a["assistant_id"], a["graph_id"], a.get("description"))

## 2. 流式调用 agent（彩色实时打印）

In [ ]:
import json
from datetime import datetime

class Colors:
    BLUE = "\033[94m"
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

async def stream_agent_platform(query: str):
    """通过 LangGraph Platform 流式运行 paas-agent，并实时打印事件。"""
    print("=" * 70)
    print(f"{Colors.BOLD}任务: {query}{Colors.RESET}")
    print(f"时间: {datetime.now().strftime('%H:%M:%S')}")
    print("=" * 70)
    print()

    in_thinking = False
    final_answer = ""

    async for part in client.runs.stream(
        thread_id=None,
        assistant_id=ASSISTANT_ID,
        input={
            "messages": [{"type": "human", "content": query}],
            "task_status": "incomplete",
        },
        stream_mode=["values", "events"],
        config={"recursion_limit": 30},
    ):
        # part 是 StreamPart(event=..., data=..., id=...)
        event_name = part.event
        data = part.data

        if event_name == "events":
            inner = data.get("event", "")
            name = data.get("name", "")
            event_data = data.get("data", {})

            if inner == "on_chat_model_stream":
                chunk = event_data.get("chunk", {})
                content = chunk.get("content", "") if isinstance(chunk, dict) else getattr(chunk, "content", "")
                if content:
                    if not in_thinking:
                        print(f"{Colors.BLUE}💭 ", end="")
                        in_thinking = True
                    print(content, end="", flush=True)

            elif inner == "on_tool_start":
                if in_thinking:
                    print()
                    in_thinking = False
                tool_name = event_data.get("tool", name) or "工具"
                print(f"\n{Colors.YELLOW}⚡ 执行工具: {tool_name}{Colors.RESET}")

            elif inner == "on_tool_end":
                output = event_data.get("output", "")
                try:
                    parsed = json.loads(output) if isinstance(output, str) else output
                    display = json.dumps(parsed, ensure_ascii=False)
                except Exception:
                    display = str(output)
                print(f"{Colors.GREEN}✅ 工具返回: {display}{Colors.RESET}\n")

            elif inner == "on_chain_start" and name == "call_model":
                print(f"\n{Colors.BLUE}┌─ 模型思考...{Colors.RESET}")

            elif inner == "on_chain_end" and name == "call_model":
                if in_thinking:
                    print()
                    in_thinking = False
                print(f"{Colors.BLUE}└─ 思考完成{Colors.RESET}\n")

        elif event_name == "values":
            # 每次状态更新时，最后一条 AI 消息就是当前最新回答
            messages = data.get("messages", [])
            if messages:
                last = messages[-1]
                if last.get("type") == "ai" and last.get("content"):
                    final_answer = last["content"]

    print("=" * 70)
    print(f"{Colors.GREEN}✅ 执行完成{Colors.RESET}")
    print(f"最终回答: {final_answer[:200]}{'...' if len(final_answer) > 200 else ''}")
    print("=" * 70)

In [ ]:
await stream_agent_platform("计算 2 + 3")

In [ ]:
await stream_agent_platform("计算 (128 + 256) / 4，然后查询北京天气")

## 3. 原始 HTTP / SSE 请求（不依赖 SDK）

In [ ]:
import requests
import json

url = f"{API_URL}/runs/stream"
payload = {
    "assistant_id": ASSISTANT_ID,
    "input": {
        "messages": [{"type": "human", "content": "计算 23 * 47"}],
        "task_status": "incomplete"
    },
    "stream_mode": ["values", "events"],
    "config": {"recursion_limit": 30}
}

with requests.post(url, json=payload, stream=True) as resp:
    print(f"状态码: {resp.status_code}")
    for line in resp.iter_lines():
        if not line:
            continue
        text = line.decode("utf-8")
        if not text.startswith("data:"):
            continue
        data_json = text[len("data:"):].strip()
        try:
            part = json.loads(data_json)
            # 只打印模型 token，避免输出过长
            if part.get("event") == "events":
                inner = part["data"].get("event", "")
                if inner == "on_chat_model_stream":
                    content = part["data"].get("data", {}).get("chunk", {}).get("content", "")
                    if content:
                        print(content, end="", flush=True)
        except json.JSONDecodeError:
            pass
print()

In [ ]:
## 4. 复刻前端 parseEvent 的完整解析器

把 `useAgentChat.ts` 里的 `parseEvent` 逻辑翻译成 Python，边收流式事件边构建出和前端一样的 `ReactCycle / ThinkStep / ToolStep` 结构，同时保留原始事件样本用于对比。

from dataclasses import dataclass, field
from typing import List, Optional, Any
from datetime import datetime
import json


@dataclass
class ThinkStep:
    type: str = "think"
    title: str = "思考过程"
    content: str = ""
    status: str = "loading"


@dataclass
class ToolStep:
    type: str = "tool"
    title: str = ""
    name: str = ""
    input: Optional[str] = None
    output: Optional[str] = None
    status: str = "loading"


@dataclass
class ReactCycle:
    id: str
    steps: List[Any] = field(default_factory=list)
    finalContent: Optional[str] = None
    status: str = "loading"


@dataclass
class ParserState:
    cycles: List[ReactCycle] = field(default_factory=list)
    currentCycle: Optional[ReactCycle] = None
    toolStack: List[ToolStep] = field(default_factory=list)
    finalOutput: str = ""
    lastModelContent: str = ""
    rootActive: bool = False
    reflectBuffer: str = ""
    reflectActive: bool = False
    processedChainEvents: set = field(default_factory=set)
    hasReactNodes: bool = False
    streamingOutput: str = ""


_uid_counter = 0


def _uid(prefix: str = "cycle") -> str:
    global _uid_counter
    _uid_counter += 1
    return f"{prefix}-{_uid_counter}-{datetime.now().strftime('%H%M%S%f')[:-3]}"


def _format_tool_output(output: Any) -> str:
    if output is None:
        return ""
    if isinstance(output, str):
        try:
            parsed = json.loads(output)
            return json.dumps(parsed, ensure_ascii=False, indent=2)
        except Exception:
            return output
    return json.dumps(output, ensure_ascii=False, indent=2)


def _as_dict(obj: Any) -> dict:
    """兼容 SDK 返回的 dict / Pydantic model。"""
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "dict"):
        return obj.dict()
    raise TypeError(f"Unsupported event data type: {type(obj)}")


def parse_event(p: ParserState, chunk: dict):
    """与前端 useAgentChat.ts 中 parseEvent 对等的实现。"""
    event = chunk.get("event", "")
    name = chunk.get("name", "")
    data = _as_dict(chunk.get("data", {}) or {})
    parent_ids = chunk.get("parent_ids", []) or []
    run_id = chunk.get("run_id", "")
    is_root = len(parent_ids) == 0
    is_react_node = name.startswith("react_")

    # 去重：LangGraph v2 会重复发送 chain 事件
    if event in ("on_chain_start", "on_chain_end"):
        signature = f"{event}|{name}|{run_id}|{','.join(parent_ids)}"
        if signature in p.processedChainEvents:
            return
        p.processedChainEvents.add(signature)

    if event == "on_chain_start":
        if is_root:
            p.rootActive = True
        elif is_react_node:
            p.hasReactNodes = True
            p.currentCycle = ReactCycle(id=_uid("cycle"))
            p.cycles.append(p.currentCycle)
        elif name == "call_model":
            if not p.hasReactNodes:
                if p.currentCycle and p.currentCycle.status == "loading":
                    p.currentCycle.status = "success"
                p.currentCycle = None
            if not p.currentCycle or p.currentCycle.status != "loading":
                p.currentCycle = ReactCycle(id=_uid("cycle"))
                p.cycles.append(p.currentCycle)
            pending_reflect = p.reflectBuffer
            p.reflectBuffer = ""
            p.streamingOutput = ""
            p.finalOutput = ""
            step = ThinkStep(content=pending_reflect)
            p.currentCycle.steps.append(step)
        elif name == "reflect":
            p.reflectActive = True
            p.reflectBuffer = ""
        return

    if event == "on_chain_end":
        if is_root:
            p.rootActive = False
            out_messages = (
                data.get("output", {}).get("messages", [])
                if isinstance(data.get("output"), dict)
                else []
            )
            if out_messages:
                p.finalOutput = out_messages[-1].get("content", "")
        elif is_react_node:
            if p.currentCycle:
                p.currentCycle.status = "success"
                p.currentCycle = None
        elif name == "call_model":
            if p.currentCycle:
                out_messages = (
                    data.get("output", {}).get("messages", [])
                    if isinstance(data.get("output"), dict)
                    else []
                )
                think_steps = [
                    s
                    for s in p.currentCycle.steps
                    if isinstance(s, ThinkStep) and s.status == "loading"
                ]
                think_step = think_steps[0] if think_steps else None

                if out_messages:
                    last = out_messages[-1]
                    has_tool_calls = (
                        isinstance(last.get("tool_calls"), list)
                        and len(last["tool_calls"]) > 0
                    )
                    has_content = (
                        isinstance(last.get("content"), str)
                        and last["content"].strip()
                    )

                    if has_content:
                        p.lastModelContent = last["content"]
                        p.currentCycle.finalContent = last["content"]
                        p.currentCycle.status = "success"
                        if think_step:
                            think_step.status = "success"
                        p.streamingOutput = ""
                        p.finalOutput = last["content"]
                        return

                    if think_step and has_tool_calls and not think_step.content.strip():
                        idx = p.currentCycle.steps.index(think_step)
                        if idx >= 0:
                            p.currentCycle.steps.pop(idx)
                        p.streamingOutput = ""
                        p.finalOutput = ""
                        return

                if think_step:
                    think_step.status = "success"
                p.streamingOutput = ""
                p.finalOutput = ""
        elif name == "reflect":
            p.reflectActive = False
        return

    if event == "on_chat_model_stream":
        chunk_data = _as_dict(data.get("chunk", {}) or {})
        content = chunk_data.get("content", "")
        if not content:
            return
        if p.reflectActive:
            p.reflectBuffer += content
        elif p.currentCycle:
            think_steps = [
                s
                for s in p.currentCycle.steps
                if isinstance(s, ThinkStep) and s.status == "loading"
            ]
            think_step = think_steps[0] if think_steps else None
            if think_step:
                think_step.content += content
                p.streamingOutput += content
                p.finalOutput = p.streamingOutput
        return

    if event == "on_tool_start":
        if p.currentCycle:
            tool_name = data.get("tool", name) or "工具"
            tool_input = data.get("input")
            step = ToolStep(
                title=tool_name,
                name=tool_name,
                input=json.dumps(tool_input, ensure_ascii=False, indent=2)
                if tool_input is not None
                else None,
            )
            p.toolStack.append(step)
            p.currentCycle.steps.append(step)
        return

    if event == "on_tool_end":
        step = p.toolStack.pop() if p.toolStack else None
        if step:
            step.output = _format_tool_output(data.get("output"))
            step.status = "success"
        return


async def stream_and_parse(query: str, max_raw_samples: int = 30):
    """流式调用 agent，返回解析状态和前 max_raw_samples 条原始事件。"""
    p = ParserState()
    raw_samples = []
    async for part in client.runs.stream(
        thread_id=None,
        assistant_id=ASSISTANT_ID,
        input={
            "messages": [{"type": "human", "content": query}],
            "task_status": "incomplete",
        },
        stream_mode=["values", "events"],
        config={"recursion_limit": 30},
    ):
        part_event = getattr(part, "event", None)
        part_data = getattr(part, "data", None)
        if part_event == "events":
            chunk = _as_dict(part_data)
            if len(raw_samples) < max_raw_samples:
                raw_samples.append(chunk)
            parse_event(p, chunk)
        elif part_event == "values":
            values = _as_dict(part_data)
            messages = values.get("messages", [])
            if messages:
                last = messages[-1]
                if isinstance(last, dict) and last.get("type") == "ai" and last.get("content"):
                    p.finalOutput = last["content"]
    return p, raw_samples


p, raw_samples = await stream_and_parse("计算 2 + 3")

print("=" * 70)
print("原始事件样本（events 流，前 %d 条）" % len(raw_samples))
print("=" * 70)
for i, e in enumerate(raw_samples):
    print(
        f"{i:2d}: event={e.get('event'):22s} name={e.get('name'):20s} "
        f"run_id={str(e.get('run_id',''))[:8]:8s} parent_ids={e.get('parent_ids', [])}"
    )

print("\n" + "=" * 70)
print("关键事件完整 JSON（chain / tool / model）")
print("=" * 70)
for e in raw_samples:
    ev_name = e.get("event", "")
    if ev_name in ("on_chain_start", "on_chain_end", "on_tool_start", "on_tool_end"):
        print(json.dumps(e, ensure_ascii=False, indent=2))
        print("-" * 40)

print("\n" + "=" * 70)
print("解析后的 ReactCycle 结构（与前端 state 等价）")
print("=" * 70)
for c in p.cycles:
    print(f"\nCycle {c.id}  status={c.status}  finalContent={c.finalContent!r}")
    for s in c.steps:
        if isinstance(s, ThinkStep):
            preview = s.content[:120].replace('\n', ' ')
            print(f"  [think] {preview}{'...' if len(s.content) > 120 else ''}  status={s.status}")
        else:
            inp = (s.input or "")[:80].replace('\n', ' ')
            out = (s.output or "")[:80].replace('\n', ' ')
            print(f"  [tool]  name={s.name}")
            print(f"          input={inp}{'...' if len(s.input or '') > 80 else ''}")
            print(f"          output={out}{'...' if len(s.output or '') > 80 else ''}  status={s.status}")

print("\n" + "=" * 70)
print("finalOutput（最后一条 AI 消息）")
print("=" * 70)
print(p.finalOutput)


In [ ]:
## 5. 直接测试 agent 全部工具

不经过 LLM，直接调用 `paas_core.agent.tools` 里的每个工具，验证输入输出和参数 schema。

import sys
sys.path.insert(0, "/Users/lanzhengpeng/develop/thesis/paas_project")

from paas_core.agent.tools import calculator, get_weather, search, query_database

print("=" * 60)
print("测试 agent 工具集合")
print("=" * 60)

test_cases = [
    ("calculator", lambda: calculator.invoke({"expression": "(128 + 256) / 4"})),
    ("calculator_error", lambda: calculator.invoke({"expression": "1 + __import__('os').system('ls')"})),
    ("get_weather", lambda: get_weather.invoke({"city": "北京"})),
    ("search", lambda: search.invoke({"query": "LangGraph 流式事件"})),
    ("query_database", lambda: query_database.invoke({"sql": "SELECT * FROM users LIMIT 5"})),
]

for name, fn in test_cases:
    print(f"\n工具: {name}")
    print("-" * 40)
    try:
        result = fn()
        print(f"结果: {result}")
    except Exception as e:
        print(f"异常: {type(e).__name__}: {e}")

print("\n" + "=" * 60)
print("工具元信息")
print("=" * 60)
for tool in [calculator, get_weather, search, query_database]:
    print(f"\n{tool.name}: {tool.description.strip()[:120]}...")
    print(f"  args schema: {tool.args}")
